# Inverse Document Frecuency
**Johann Pasquel**

## 1. No library solution

In [4]:
from pathlib import Path
from collections import Counter
import csv
import random
import re
import math

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from IPython.display import display

root_dir = Path.cwd().resolve()

if root_dir.name == "notebooks":
    root_dir = root_dir.parent

ruta_nltk = root_dir / "data" / "nltk_data"

if str(ruta_nltk) not in nltk.data.path:
    nltk.data.path.insert(0, str(ruta_nltk))

try:
    palabras_vacias = set(stopwords.words("english"))
except LookupError:
    nltk.download(
        "stopwords",
        download_dir=str(ruta_nltk),
        raise_on_error=True,
    )
    palabras_vacias = set(stopwords.words("english"))

print("Carpeta del proyecto:", root_dir)
print("Stopwords cargadas:", len(palabras_vacias))

Carpeta del proyecto: C:\Users\johan\Desktop\UNI\7 Semestre\Recuperación_de_Información
Stopwords cargadas: 198


#### 100 documentos

In [5]:
archivo_inventario = (
    root_dir / "data" / "large_gutenberg" / "documentos_validos.csv"
)

with archivo_inventario.open(encoding="utf-8", newline="") as entrada:
    inventario = list(csv.DictReader(entrada))

N = 100

if len(inventario) < N:
    raise ValueError("El inventario no contiene 100 documentos.")

seleccion = random.Random(42).sample(inventario, N)

print("Documentos seleccionados:", len(seleccion))
display(pd.DataFrame(seleccion).head(10))

Documentos seleccionados: 100


,doc_id,ruta,bytes
0,15477,data/large_gutenberg/txt/cache/epub/15477/pg15...,2621948
1,3448,data/large_gutenberg/txt/cache/epub/3448/pg344...,800910
2,37573,data/large_gutenberg/txt/cache/epub/37573/pg37...,1032978
3,33595,data/large_gutenberg/txt/cache/epub/33595/pg33...,115816
4,30726,data/large_gutenberg/txt/cache/epub/30726/pg30...,904866
5,19186,data/large_gutenberg/txt/cache/epub/19186/pg19...,527224
6,14313,data/large_gutenberg/txt/cache/epub/14313/pg14...,684325
7,73185,data/large_gutenberg/txt/cache/epub/73185/pg73...,83151
8,12231,data/large_gutenberg/txt/cache/epub/12231/pg12...,103116
9,79114,data/large_gutenberg/txt/cache/epub/79114/pg79...,347176


#### Procesamiento y vocabulario

In [6]:
def preprocesar(texto):
    palabras = re.findall(r"[^\W\d_]+", texto.lower())
    return [
        palabra
        for palabra in palabras
        if palabra not in palabras_vacias
    ]


conteos = {}
resumen = []

for numero, registro in enumerate(seleccion, start=1):
    doc_id = registro["doc_id"]
    ruta = root_dir / registro["ruta"]

    texto = ruta.read_text(encoding="utf-8-sig")
    tokens = preprocesar(texto)

    conteos[doc_id] = Counter(tokens)

    resumen.append({
        "Documento": doc_id,
        "Tokens": len(tokens)
    })

    if numero % 10 == 0:
        print(f"Documentos procesados: {numero}/{N}")

documentos = list(conteos)
vocabulario = sorted(set().union(*conteos.values()))

posicion = {
    termino: columna
    for columna, termino in enumerate(vocabulario)
}

print("\nDocumentos:", len(documentos))
print("Términos del vocabulario:", len(vocabulario))

display(pd.DataFrame(resumen).head(10))

Documentos procesados: 10/100
Documentos procesados: 20/100
Documentos procesados: 30/100
Documentos procesados: 40/100
Documentos procesados: 50/100
Documentos procesados: 60/100
Documentos procesados: 70/100
Documentos procesados: 80/100
Documentos procesados: 90/100
Documentos procesados: 100/100

Documentos: 100
Términos del vocabulario: 212102


,Documento,Tokens
0,15477,228748
1,3448,71332
2,37573,87254
3,33595,16452
4,30726,79775
5,19186,84329
6,14313,56178
7,73185,9861
8,12231,8564
9,79114,29156


#### Construcción de la Matriz TF

In [7]:
matriz_tf = np.zeros(
    (len(documentos), len(vocabulario)),
    dtype=np.int32,
)

for i, doc_id in enumerate(documentos):
    for termino, frecuencia in conteos[doc_id].items():
        j = posicion[termino]
        matriz_tf[i, j] = frecuencia

tabla_tf = pd.DataFrame(
    matriz_tf,
    index=documentos,
    columns=vocabulario,
    copy=False,
)

tabla_tf.index.name = "Documento"
tabla_tf.columns.name = "Término"

print("Dimensiones de TF:", matriz_tf.shape)
print("Muestra: 10 documentos y 15 términos")
display(tabla_tf.iloc[:10, :15])


Dimensiones de TF: (100, 212102)
Muestra: 10 documentos y 15 términos


Término,aa,aabraham,aabrahamin,aachen,aadam,aadamin,aadlaars,aakkosia,aakkosista,aakkosten,aal,aalbessestruik,aallolle,aallon,aalst
Documento,,,,,,,,,,,,,,,
15477,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3448,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
37573,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
33595,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
30726,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
19186,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
14313,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
73185,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
12231,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
def element_wise_multiplication(matrix_idf, vector_idf):
    """
    Perform element-wise multiplication of a matrix and a vector.

    Args:
        matrix_idf (list of list of float): A 2D list representing the matrix.
        vector_idf (list of float): A 1D list representing the vector.
    """
    # Check if the number of columns in the matrix matches the length of the vector
    if len(matrix_idf[0]) != len(vector_idf):
        raise ValueError("The number of columns in the matrix must match the length of the vector.")

    # Perform element-wise multiplication
    result = []
    for row in matrix_idf:
        multiplied_row = [row[i] * vector_idf[i] for i in range(len(row))]
        result.append(multiplied_row)

    return result

#### Calcular DF e IDF

In [9]:
vector_df = np.zeros(len(vocabulario), dtype=np.int32)

for j in range(len(vocabulario)):
    vector_df[j] = sum(
        1
        for i in range(len(documentos))
        if matriz_tf[i, j] > 0
    )

vector_idf = np.array([
    math.log10(N / df)
    for df in vector_df
])

tabla_idf = pd.DataFrame({
    "Término": vocabulario,
    "DF": vector_df,
    "IDF": vector_idf,
}).set_index("Término")

print("Cantidad de valores IDF:", len(vector_idf))
display(tabla_idf.head(15).round(5))

Cantidad de valores IDF: 212102


,DF,IDF
Término,,
aa,1,2.00000
aabraham,1,2.00000
aabrahamin,1,2.00000
aachen,2,1.69897
aadam,1,2.00000
aadamin,1,2.00000
aadlaars,1,2.00000
aakkosia,1,2.00000
aakkosista,1,2.00000


#### Element-wise Multiplication

In [10]:
def element_wise_multiplication(matrix_tf, vector_idf):
    """
    Multiplica cada celda de TF por el IDF de su columna.
    """
    if len(matrix_tf) == 0 or len(vector_idf) == 0:
        raise ValueError("La matriz y el vector deben contener datos.")

    if any(len(fila) != len(vector_idf) for fila in matrix_tf):
        raise ValueError(
            "Cada fila debe tener tantas columnas como valores IDF."
        )

    resultado = np.zeros(
        (len(matrix_tf), len(vector_idf)),
        dtype=float,
    )

    for i, fila in enumerate(matrix_tf):
        for j, frecuencia in enumerate(fila):
            resultado[i, j] = frecuencia * vector_idf[j]

    return resultado

#### Obtener Matriz TF-IDF

In [11]:
matriz_tfidf = element_wise_multiplication(
    matriz_tf,
    vector_idf,
)

tabla_tfidf = pd.DataFrame(
    matriz_tfidf,
    index=documentos,
    columns=vocabulario,
    copy=False,
)

tabla_tfidf.index.name = "Documento"
tabla_tfidf.columns.name = "Término"

print("Dimensiones de TF-IDF:", matriz_tfidf.shape)
display(tabla_tfidf.iloc[:10, :15].round(4))

Dimensiones de TF-IDF: (100, 212102)


Término,aa,aabraham,aabrahamin,aachen,aadam,aadamin,aadlaars,aakkosia,aakkosista,aakkosten,aal,aalbessestruik,aallolle,aallon,aalst
Documento,,,,,,,,,,,,,,,
15477,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3448,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
37573,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33595,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30726,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19186,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
73185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12231,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Cálculo

In [12]:
doc_id = documentos[0]

print(f"Ejemplos del documento {doc_id}\n")

for termino, frecuencia in conteos[doc_id].most_common(5):
    j = posicion[termino]
    df = vector_df[j]
    idf = vector_idf[j]
    peso = matriz_tfidf[0, j]

    print(f"Término: {termino}")
    print(f"TF = {frecuencia}")
    print(f"DF = {df}")
    print(f"IDF = log10({N}/{df}) = {idf:.5f}")
    print(f"TF-IDF = {frecuencia} × {idf:.5f} = {peso:.4f}")
    print()

Ejemplos del documento 15477

Término: thou
TF = 3752
DF = 30
IDF = log10(100/30) = 0.52288
TF-IDF = 3752 × 0.52288 = 1961.8411

Término: one
TF = 3234
DF = 100
IDF = log10(100/100) = 0.00000
TF-IDF = 3234 × 0.00000 = 0.0000

Término: said
TF = 2160
DF = 72
IDF = log10(100/72) = 0.14267
TF-IDF = 2160 × 0.14267 = 308.1618

Término: king
TF = 1688
DF = 56
IDF = log10(100/56) = 0.25181
TF-IDF = 1688 × 0.25181 = 425.0586

Término: great
TF = 1653
DF = 73
IDF = log10(100/73) = 0.13668
TF-IDF = 1653 × 0.13668 = 225.9273



## 2. Library Solution

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

documentos_sklearn = []
textos_sklearn = []

for registro in seleccion:
    ruta = root_dir / registro["ruta"]

    documentos_sklearn.append(registro["doc_id"])
    textos_sklearn.append(
        ruta.read_text(encoding="utf-8-sig")
    )

print("Documentos cargados:", len(textos_sklearn))

Documentos cargados: 100


In [17]:
vectorizador = TfidfVectorizer(
    analyzer=preprocesar,
    norm=None,
    smooth_idf=False,
    sublinear_tf=False,
)

matriz_tfidf_sklearn = vectorizador.fit_transform(textos_sklearn)

terminos_sklearn = vectorizador.get_feature_names_out()
idf_sklearn = vectorizador.idf_

print("Documentos:", matriz_tfidf_sklearn.shape[0])
print("Términos:", matriz_tfidf_sklearn.shape[1])
print("Dimensiones de TF-IDF:", matriz_tfidf_sklearn.shape)

Documentos: 100
Términos: 212102
Dimensiones de TF-IDF: (100, 212102)


In [18]:
tabla_idf_sklearn = pd.DataFrame({
    "Término": terminos_sklearn,
    "IDF sklearn": idf_sklearn,
}).set_index("Término")

print("Muestra de los primeros 15 términos:")
display(tabla_idf_sklearn.head(15).round(5))

Muestra de los primeros 15 términos:


,IDF sklearn
Término,
aa,5.60517
aabraham,5.60517
aabrahamin,5.60517
aachen,4.91202
aadam,5.60517
aadamin,5.60517
aadlaars,5.60517
aakkosia,5.60517
aakkosista,5.60517


In [19]:
tabla_muestra_sklearn = pd.DataFrame(
    matriz_tfidf_sklearn[:10, :15].toarray(),
    index=documentos_sklearn[:10],
    columns=terminos_sklearn[:15],
)

tabla_muestra_sklearn.index.name = "Documento"
tabla_muestra_sklearn.columns.name = "Término"

display(tabla_muestra_sklearn.round(4))

Término,aa,aabraham,aabrahamin,aachen,aadam,aadamin,aadlaars,aakkosia,aakkosista,aakkosten,aal,aalbessestruik,aallolle,aallon,aalst
Documento,,,,,,,,,,,,,,,
15477,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3448,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
37573,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33595,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30726,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19186,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
73185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12231,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
fila = matriz_tfidf_sklearn.getrow(0)

pesos_documento = pd.DataFrame({
    "Término": terminos_sklearn[fila.indices],
    "TF-IDF": fila.data,
}).sort_values("TF-IDF", ascending=False)

print("Documento:", documentos_sklearn[0])
display(pesos_documento.head(10).round(4))

Documento: 15477


,Término,TF-IDF
147,thou,8269.3060
369,brahmanas,4758.7895
628,brahmana,4730.7636
91,yudhishthira,3648.9658
711,deities,3611.3577
109,thee,3288.3274
240,one,3234.0000
746,endued,3020.7736
100,thy,2965.8360
92,said,2869.5688
